In [ ]:
# ETAPA 1 — Bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
sns.set_palette('deep')

In [ ]:
# ETAPA 2 — Carregar CSV e converter data
df = pd.read_csv('vendas.csv')
df['data'] = pd.to_datetime(df['data'])
display(df.head())

In [ ]:
# ETAPA 3 — Faturamento = quantidade × preço unitário
df['faturamento'] = df['quantidade'] * df['preco_unitario']
display(df.head())

In [ ]:
# ETAPA 4 — Integridade do dataset
df.info()
print('\nValores ausentes:')
print(df.isnull().sum())
print('\nResumo estatístico:')
display(df.describe(include='all'))

In [ ]:
# ETAPA 5 — Substituir valores nulos pela média da coluna
df['quantidade'] = df['quantidade'].fillna(df['quantidade'].mean())
df['preco_unitario'] = df['preco_unitario'].fillna(df['preco_unitario'].mean())
df['faturamento'] = df['quantidade'] * df['preco_unitario']
print(df.isnull().sum())

In [ ]:
# ETAPA 6 — Faturamento total por loja, maior para menor
faturamento_loja = df.groupby('loja')['faturamento'].sum().sort_values(ascending=False)
display(faturamento_loja)

In [ ]:
# ETAPA 7 — Ticket médio por loja
ticket = df.groupby('loja').agg(
    faturamento=('faturamento','sum'),
    numero_vendas=('loja','size')
)
ticket['ticket_medio'] = ticket['faturamento'] / ticket['numero_vendas']
display(ticket.sort_values('ticket_medio', ascending=False))

In [ ]:
# ETAPA 8 — Cinco produtos mais vendidos em quantidade
top5 = df.groupby('produto')['quantidade'].sum().sort_values(ascending=False).head(5)
display(top5)

In [ ]:
# ETAPA 9 — Mês de maior e menor faturamento
resumo_mensal = df.groupby(df['data'].dt.to_period('M'))['faturamento'].sum()
maior = resumo_mensal.idxmax()
menor = resumo_mensal.idxmin()
print(f'Maior: {maior.strftime("%m/%Y")} — R$ {resumo_mensal.max():,.2f}')
print(f'Menor: {menor.strftime("%m/%Y")} — R$ {resumo_mensal.min():,.2f}')

In [ ]:
# ETAPA 10 — Coluna mês
df['mes'] = df['data'].dt.month
display(df[['data','mes']].head())

In [ ]:
# ETAPA 11 — Barras: faturamento por loja
Path('graficos').mkdir(exist_ok=True)
plt.figure(figsize=(8,5))
sns.barplot(x=faturamento_loja.index, y=faturamento_loja.values, hue=faturamento_loja.index, legend=False)
plt.title('Faturamento Total por Loja')
plt.xlabel('Loja')
plt.ylabel('Faturamento (R$)')
plt.tight_layout()
plt.savefig('graficos/faturamento_por_loja.png', dpi=150)
plt.show()

In [ ]:
# ETAPA 12 — Linhas: evolução mensal
mensal = resumo_mensal.copy()
mensal.index = mensal.index.to_timestamp()
plt.figure(figsize=(10,5))
plt.plot(mensal.index, mensal.values, marker='o')
plt.title('Evolução Mensal do Faturamento')
plt.xlabel('Mês')
plt.ylabel('Faturamento (R$)')
plt.xticks(rotation=45)
for x,y in zip(mensal.index,mensal.values):
    plt.annotate(f'R$ {y:,.0f}', (x,y), xytext=(0,8), textcoords='offset points', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('graficos/evolucao_mensal.png', dpi=150)
plt.show()

In [ ]:
# ETAPA 13 — Pizza: faturamento por região
faturamento_regiao = df.groupby('regiao')['faturamento'].sum()
plt.figure(figsize=(7,7))
plt.pie(faturamento_regiao, labels=faturamento_regiao.index, autopct='%1.1f%%', startangle=90)
plt.title('Distribuição do Faturamento por Região')
plt.tight_layout()
plt.savefig('graficos/faturamento_por_regiao.png', dpi=150)
plt.show()

In [ ]:
# ETAPA 14 — Scatter: quantidade × faturamento
plt.figure(figsize=(8,5))
sns.scatterplot(data=df, x='quantidade', y='faturamento', hue='loja', s=80)
plt.title('Relação entre Quantidade Vendida e Faturamento')
plt.xlabel('Quantidade')
plt.ylabel('Faturamento (R$)')
plt.tight_layout()
plt.savefig('graficos/scatter_quantidade_faturamento.png', dpi=150)
plt.show()

In [ ]:
# ETAPA 15 — Boxplot: preços por categoria
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='categoria', y='preco_unitario', hue='categoria', legend=False)
plt.title('Distribuição dos Preços por Categoria')
plt.xlabel('Categoria')
plt.ylabel('Preço Unitário (R$)')
plt.tight_layout()
plt.savefig('graficos/boxplot_precos_categoria.png', dpi=150)
plt.show()

In [ ]:
# ETAPA 16 — Heatmap de correlação
correlacao = df.select_dtypes(include=np.number).corr()
plt.figure(figsize=(9,6))
sns.heatmap(correlacao, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de Correlação das Variáveis Numéricas')
plt.tight_layout()
plt.savefig('graficos/heatmap_correlacao.png', dpi=150)
plt.show()

In [ ]:
# ETAPA 18 — Títulos, eixos e legibilidade
plt.figure(figsize=(8,5))
sns.barplot(x=faturamento_loja.index, y=faturamento_loja.values, hue=faturamento_loja.index, legend=False)
plt.title('Faturamento Total por Loja')
plt.xlabel('Loja')
plt.ylabel('Faturamento (R$)')
plt.tight_layout()
plt.show()

In [ ]:
# ETAPA 19 — Layout com múltiplos gráficos
fig, axes = plt.subplots(1,2,figsize=(13,5))
sns.barplot(x=faturamento_loja.index, y=faturamento_loja.values, hue=faturamento_loja.index, legend=False, ax=axes[0])
axes[0].set_title('Faturamento por Loja')
axes[0].set_xlabel('Loja')
axes[0].set_ylabel('Faturamento (R$)')
sns.barplot(x=faturamento_regiao.index, y=faturamento_regiao.values, hue=faturamento_regiao.index, legend=False, ax=axes[1])
axes[1].set_title('Faturamento por Região')
axes[1].set_xlabel('Região')
axes[1].set_ylabel('Faturamento (R$)')
plt.tight_layout()
plt.savefig('graficos/layout_comparativo.png', dpi=150)
plt.show()

In [ ]:
# ETAPA 20 — Exportação e relatório final
resumo_analitico = pd.DataFrame({
    'faturamento_total':[df['faturamento'].sum()],
    'loja_maior_faturamento':[faturamento_loja.idxmax()],
    'maior_faturamento_loja':[faturamento_loja.max()],
    'loja_menor_faturamento':[faturamento_loja.idxmin()],
    'menor_faturamento_loja':[faturamento_loja.min()],
    'mes_maior_faturamento':[maior.strftime('%m/%Y')],
    'maior_faturamento_mes':[resumo_mensal.max()],
    'mes_menor_faturamento':[menor.strftime('%m/%Y')],
    'menor_faturamento_mes':[resumo_mensal.min()]
})
resumo_analitico.to_csv('resumo_analitico.csv', index=False)
df.to_csv('vendas_processadas.csv', index=False)
display(resumo_analitico)